# Analisi Esplorativa dei Dati CNI

Questo notebook analizza i dati crawlti dal sito del Consiglio Nazionale degli Ingegneri (CNI) per comprendere:
- Distribuzione delle pagine per categoria
- Lunghezza dei contenuti
- Qualità del testo
- Copertura delle sezioni del sito

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import json
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from collections import Counter
from src.ingestion.downloader import Downloader

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

print('Setup completato')

In [ ]:
# Carica documenti crawlti
downloader = Downloader(output_dir="data/raw")
documents = downloader.load_documents()
print(f"Documenti caricati: {len(documents)}")

In [ ]:
# Analisi: distribuzione per categoria (dal classificatore)
from src.rag.query_classifier import QueryClassifier
qc = QueryClassifier()

categories = []
for doc in documents:
    url = doc.get("url", "")
    content = doc.get("content", "")
    cat = qc.classify(content[:200])
    categories.append(cat)

cat_counts = Counter(categories)
df_cats = pd.DataFrame(cat_counts.items(), columns=["Categoria", "Conteggio"]).sort_values("Conteggio", ascending=False)

fig, ax = plt.subplots()
bars = ax.bar(df_cats["Categoria"], df_cats["Conteggio"], color=sns.color_palette("husl", len(df_cats)))
ax.set_title("Distribuzione Documenti per Categoria", fontsize=16, fontweight='bold')
ax.set_xlabel("Categoria")
ax.set_ylabel("Numero Documenti")
for bar, count in zip(bars, df_cats["Conteggio"]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, str(count), ha='center', fontsize=10)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Analisi: distribuzione lunghezza contenuti
lengths = [len(doc.get("content", "")) for doc in documents]
df_len = pd.DataFrame({"lunghezza": lengths})

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(lengths, bins=30, edgecolor='black', alpha=0.7, color='steelblue')
axes[0].axvline(df_len["lunghezza"].median(), color='red', linestyle='--', label=f'Mediana: {df_len["lunghezza"].median():.0f}')
axes[0].set_title("Distribuzione Lunghezza Contenuti", fontsize=14, fontweight='bold')
axes[0].set_xlabel("Numero Caratteri")
axes[0].set_ylabel("Frequenza")
axes[0].legend()

axes[1].boxplot(lengths, vert=False, patch_artist=True, boxprops=dict(facecolor='steelblue'))
axes[1].set_title("Boxplot Lunghezza Contenuti", fontsize=14, fontweight='bold')
axes[1].set_xlabel("Numero Caratteri")

plt.tight_layout()
plt.show()

print(f"Statistiche:")
print(f"  Media: {df_len['lunghezza'].mean():.0f}")
print(f"  Mediana: {df_len['lunghezza'].median():.0f}")
print(f"  Min: {df_len['lunghezza'].min()}")
print(f"  Max: {df_len['lunghezza'].max()}")
print(f"  Dev Std: {df_len['lunghezza'].std():.0f}")

In [ ]:
# Analisi: qualità del testo (repetition ratio)
from src.governance.quality_check import QualityChecker

qc_check = QualityChecker()
quality_flags = []
repetition_ratios = []

for doc in documents:
    ok, issues = qc_check.check(doc.get("content", ""))
    quality_flags.append(ok)
    ratio = qc_check._compute_repetition_ratio(doc.get("content", ""))
    repetition_ratios.append(ratio)

df_qual = pd.DataFrame({
    "qualità_ok": quality_flags,
    "repetition_ratio": repetition_ratios
})

fig, ax = plt.subplots()
colors = ['green' if x else 'red' for x in quality_flags]
ax.scatter(range(len(documents)), repetition_ratios, c=colors, alpha=0.6, s=30)
ax.axhline(y=0.3, color='orange', linestyle='--', label='Soglia (0.3)')
ax.set_title("Quality Check: Repetition Ratio per Documento", fontsize=14, fontweight='bold')
ax.set_xlabel("Documento")
ax.set_ylabel("Repetition Ratio")
ax.legend()
plt.tight_layout()
plt.show()

print(f"Documenti che superano quality check: {sum(quality_flags)}/{len(quality_flags)}")
print(f"Repetition ratio medio: {df_qual['repetition_ratio'].mean():.3f}")

In [ ]:
# Analisi: chunking simulation
from src.ingestion.chunker import DocumentChunker

chunker = DocumentChunker()
all_chunks = chunker.chunk_documents(documents)

chunk_lengths = [len(c["content"]) for c in all_chunks]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(chunk_lengths, bins=30, edgecolor='black', alpha=0.7, color='coral')
axes[0].axvline(512, color='red', linestyle='--', label='Chunk size target (512)')
axes[0].set_title("Distribuzione Lunghezza Chunk", fontsize=14, fontweight='bold')
axes[0].set_xlabel("Numero Caratteri")
axes[0].set_ylabel("Frequenza")
axes[0].legend()

docs_with_chunks = {}
for c in all_chunks:
    src = c["metadata"]["source"]
    docs_with_chunks[src] = docs_with_chunks.get(src, 0) + 1

axes[1].hist(list(docs_with_chunks.values()), bins=20, edgecolor='black', alpha=0.7, color='seagreen')
axes[1].set_title("Numero di Chunk per Documento", fontsize=14, fontweight='bold')
axes[1].set_xlabel("Numero Chunk")
axes[1].set_ylabel("Frequenza")

plt.tight_layout()
plt.show()

print(f"Totale chunk: {len(all_chunks)}")
print(f"Media lunghezza chunk: {pd.Series(chunk_lengths).mean():.0f}")
print(f"Documenti con più chunk: {max(docs_with_chunks.values())}")

In [ ]:
# Top domini/sezioni coperte
from urllib.parse import urlparse

paths = [urlparse(doc["url"]).path for doc in documents if "url" in doc]
sections = [p.split("/")[1] if p and len(p.split("/")) > 1 else "root" for p in paths]
section_counts = Counter(sections)

df_sections = pd.DataFrame(section_counts.most_common(15), columns=["Sezione", "Conteggio"])

fig, ax = plt.subplots()
bars = ax.barh(df_sections["Sezione"], df_sections["Conteggio"], color=sns.color_palette("viridis", len(df_sections)))
ax.set_title("Top 15 Sezioni del Sito CNI Crawlate", fontsize=14, fontweight='bold')
ax.set_xlabel("Numero Pagine")
ax.invert_yaxis()
for bar, count in zip(bars, df_sections["Conteggio"]):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2, str(count), va='center')
plt.tight_layout()
plt.show()

In [ ]:
# Riepilogo finale
print("\n" + "="*60)
print("RIEPILOGO ANALISI ESPLORATIVA DATI CNI")
print("="*60)
print(f"\n📊 Statistiche Generali:")
print(f"  - Documenti totali: {len(documents)}")
print(f"  - Categorie rilevate: {len(cat_counts)}")
print(f"  - Sezioni coperte: {len(section_counts)}")
print(f"\n📏 Contenuti:")
print(f"  - Lunghezza media: {df_len['lunghezza'].mean():.0f} caratteri")
print(f"  - Lunghezza mediana: {df_len['lunghezza'].median():.0f} caratteri")
print(f"\n🧩 Chunking:")
print(f"  - Totale chunk: {len(all_chunks)}")
print(f"  - Media chunk per documento: {len(all_chunks)/len(documents):.1f}")
print(f"\n✅ Qualità:")
print(f"  - Documenti validi: {sum(quality_flags)}/{len(quality_flags)}")
print(f"  - Repetition ratio medio: {df_qual['repetition_ratio'].mean():.3f}")